# 콜레라 원인 발견(1)
교과서 **224~225쪽** · 부록 활동 (힌트 버전)

콜레라 환자·펌프 위치를 지도에 올려, **어느 펌프 근처 환자가 몰리는지** 찾아봅니다.  
(정답 라벨 없이 거리로 묶는 아이디어 → **비지도학습·군집과 비슷한 사고**)

| 단계 | 할 일 | 힌트 |
|---|---|---|
| 1 | 모듈·CSV 불러오기 | `pandas`, `folium`, `cdist` |
| 2 | 환자↔펌프 거리 → 가장 가까운 펌프 | `cdist`, `np.argmin` (**최솟값** 위치) |
| 3 | 지도에 환자(색)·펌프(마커) | `CircleMarker`, `Marker` |
| 4 | 해석 | 색이 몰린 펌프 = 의심 펌프 |

**준비물:** `펌프위치.csv`, `콜레라환자.csv` · `folium`, `scipy`  
```
!pip install folium scipy
```
**배경:** 1850년대 런던 소호, 존 스노(John Snow)의 콜레라 조사 자료입니다.


---

# 1. 모듈과 데이터 불러오기


In [ ]:
import numpy as np
import pandas as pd
import folium
from scipy.spatial.distance import cdist  # 점들 사이 유클리드 거리

펌프위치 = pd.read_csv('펌프위치.csv')
콜레라환자 = pd.read_csv('콜레라환자.csv')

print('펌프:', 펌프위치.shape, list(펌프위치.columns))
print('환자:', 콜레라환자.shape, list(콜레라환자.columns))
display(펌프위치.head())
display(콜레라환자.head())


> **힌트** 이 데이터에서 `경도`/`위도` 열 값은 지도용 `[위도, 경도]` 순서에 맞게 이미 정리된 경우가 많습니다.  
> 아래 코드는 교과서와 같이 `['경도','위도']` 열을 **그대로** 씁니다.


---

# 2. 데이터 전처리하기 — 가장 가까운 펌프 찾기

아이디어:
1. 환자마다 **모든 펌프까지 거리**를 계산한다. → `cdist`
2. 그중 **가장 가까운** 펌프 번호를 고른다. → `np.argmin` (**작은** 값의 위치)


In [ ]:
# 펌프 좌표만 넘파이 배열로
펌프좌표 = 펌프위치[['경도', '위도']].to_numpy()

# 환자(행) × 펌프(열) 거리 행렬
펌프와의거리 = cdist(콜레라환자[['경도', '위도']].values, 펌프좌표)
print('거리 행렬 형태:', 펌프와의거리.shape)
# → (환자 수, 펌프 수) 이어야 합니다. 예: (489, 8)

# 각 환자(행)마다 거리가 가장 작은 펌프 번호
가까운펌프번호 = np.argmin(펌프와의거리, axis=1)
print('앞 10명 환자 → 가까운 펌프 번호:', 가까운펌프번호[:10])


In [ ]:
# 펌프별로 얼마나 많은 환자가 '가장 가깝다'고 붙었는지 세기
번호, 횟수 = np.unique(가까운펌프번호, return_counts=True)
집계 = pd.DataFrame({'펌프번호': 번호, '가까운_환자수': 횟수})
집계 = 집계.sort_values('가까운_환자수', ascending=False)
집계


> **생각하기 1** `argmin`과 `argmax`의 차이는? 거리를 기준으로 가까운 펌프를 찾을 때 왜 `argmin`인가요?
> →

> **생각하기 2** 위 표에서 환자 수가 유난히 많은 펌프 번호는?
> → ____번  /  이 펌프 이름(데이터에 있다면): ____


---

# 3. 데이터 시각화하기

- 환자: 작은 **동그라미**, 가까운 펌프마다 **색**을 다르게
- 펌프: **별(마커)**


In [ ]:
colors = ['red', 'yellow', 'purple', 'deeppink', 'lime', 'aqua', 'blue', 'orange']

# 런던 소호 근처를 중심으로 지도 생성
지도 = folium.Map(location=[51.5149, -0.1348], tiles='OpenStreetMap', zoom_start=16)

# 환자 위치 표시 (행 index = 0,1,2,... 라고 가정)
for idx, row in 콜레라환자.iterrows():
    색번호 = 가까운펌프번호[idx] % len(colors)  # 펌프가 8개보다 많아도 안전하게
    folium.CircleMarker(
        location=[row['경도'], row['위도']],
        radius=5,
        color=colors[색번호],
        fill=True
    ).add_to(지도)


In [ ]:
# 펌프 위치를 별 마커로
for center in 펌프좌표:
    folium.Marker(
        location=list(center),
        icon=folium.Icon(color='blue', icon='star')
    ).add_to(지도)

display(지도)


> **생각하기 3** 지도에서 한 색의 점이 특정 펌프(별) 주변에 몰리면 무엇을 의할 수 있나요?  
> (존 스노는 오염된 **Broad Street** 펌프를 의했습니다.)
> →

---

### 제출·정리 체크
- [ ] 거리 행렬 shape = (환자 수, 펌프 수) 확인
- [ ] 펌프별 가까운 환자 수 표 또는 스크린샷
- [ ] 지도에서 의심 펌프를 한 문장으로 설명

> **심화(선택)** `펌프위치`에 이름 열이 있으면, 환자 수가 가장 많은 펌프 번호를 이름으로 바꿔 출력해 보세요.


In [ ]:
# (선택) 펌프 이름과 환자 수 연결
이름열후보 = [c for c in 펌프위치.columns if c not in ['경도', '위도']]
print('이름 후보 열:', 이름열후보)

if 이름열후보:
    이름열 = 이름열후보[0]
    결과 = 집계.copy()
    결과['펌프이름'] = 결과['펌프번호'].map(lambda i: 펌프위치.loc[i, 이름열])
    display(결과)
